In [ ]:
# Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# For nicer plots
sns.set(style="whitegrid")


In [ ]:
from rdatasets import data

# Load dataset
arrests = data("Arrests", "carData")

# Check structure
print(arrests.info())
print(arrests.head())


# Outlier Removal

In [ ]:
def remove_outliers(df):
    numeric_cols = df.select_dtypes(include=[np.number])
    outlier_mask = pd.Series(False, index=df.index)
    
    for col in numeric_cols.columns:
        q1 = numeric_cols[col].quantile(0.25)
        q3 = numeric_cols[col].quantile(0.75)
        iqr = q3 - q1
        lb = q1 - 1.5 * iqr
        ub = q3 + 1.5 * iqr
        
        outlier_mask |= (numeric_cols[col] < lb) | (numeric_cols[col] > ub)
    
    # Drop rows flagged as outliers in any numeric column
    df_clean = df.loc[~outlier_mask].reset_index(drop=True)
    return df_clean

df = remove_outliers(arrests).dropna()


# Exploratory

In [ ]:
# 1. Race vs Release
sns.countplot(data=df, x="colour", hue="released")
plt.title("How Race Affects Release Rate")
plt.show()

# 2. Citizenship vs Release
sns.countplot(data=df, x="citizen", hue="released")
plt.title("How Citizenship Affects Release Rate")
plt.show()

# 3. Age vs Release
sns.histplot(data=df, x="age", hue="released", multiple="dodge", binwidth=1)
plt.title("How Age Affects Release Rate")
plt.show()

# 4. Age vs Race
sns.histplot(data=df, x="age", hue="colour", multiple="dodge", binwidth=1)
plt.title("How Race Affects Age of Arrest")
plt.show()

# 5. Arrest record (checks) vs Release
sns.countplot(data=df, x="checks", hue="released")
plt.title("How Arrest Record Affects Release Rate")
plt.show()

# 6. Employment vs Release
sns.countplot(data=df, x="employed", hue="released")
plt.title("How Employment Status Affects Release Rate")
plt.show()


# Analysis


In [ ]:
# Analysis
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Summary of numeric and categorical data
print(df.describe(include="all"))


# Fit logistic regression
model = smf.logit("released ~ age + sex + colour + employed + citizen + checks", data=df).fit()

print(model.summary())

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = pd.get_dummies(df[["age","sex","colour","employed","citizen","checks"]], drop_first=True)
X = sm.add_constant(X)  # add intercept

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)

In [ ]:
import statsmodels.formula.api as smf

# Fit a model
model = smf.logit("released ~ age + sex + colour + employed + citizen + checks", data=df).fit()

# Get AIC
print("AIC:", model.aic)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Predictions
df["pred_prob"] = model.predict(df)
df["pred_class"] = (df["pred_prob"] >= 0.5).astype(int)

# Confusion matrix & report
print(confusion_matrix(df["released"].map({"No":0, "Yes":1}), df["pred_class"]))
print(classification_report(df["released"].map({"No":0, "Yes":1}), df["pred_class"]))

# ROC Curve
fpr, tpr, _ = roc_curve(df["released"].map({"No":0, "Yes":1}), df["pred_prob"])
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(df['released'].map({'No':0,'Yes':1}), df['pred_prob']):.3f}")
plt.plot([0,1],[0,1],"--",color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
odds_ratios = pd.DataFrame({
    "OR": model.params.apply(np.exp),
    "Lower CI": model.conf_int()[0].apply(np.exp),
    "Upper CI": model.conf_int()[1].apply(np.exp)
})
print(odds_ratios)
